In [ ]:
# ECDSA (Elliptic Curve Digital Signature Algorithm) with SHA256

import random
import subprocess
import csv
import hashlib

from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.exceptions import InvalidSignature

In [ ]:
PRIV_KEY_FILE_PATH = "private.pem"
PLUB_KEY_FILE_PATH = "public.pem"

# Generate the private key
subprocess.run(["openssl", "ecparam", "-name", "prime256v1", "-genkey", "-noout", "-out", PRIV_KEY_FILE_PATH])

# Extract the public key in PEM format
subprocess.run(["openssl", "ec", "-in", PRIV_KEY_FILE_PATH, "-pubout", "-out", PLUB_KEY_FILE_PATH], capture_output=True, text=True)

# Load Private Key
with open(PRIV_KEY_FILE_PATH, "rb") as key_file:
    openssl_private_key = serialization.load_pem_private_key(
        key_file.read(),
        password = None
    )
openssl_private_key_decoded = openssl_private_key.private_numbers().private_value
print(f"Private Key (Hex): {hex(private_key_decoded)}")

# Load Public Key
with open(PLUB_KEY_FILE_PATH, "rb") as key_file:
    key_data = key_file.read().strip()
openssl_public_key = serialization.load_pem_public_key(key_data)
openssl_public_key_decoded = openssl_public_key.public_numbers()

print(f"Public Key X: {hex(openssl_public_key_decoded.x)}")
print(f"Public Key Y: {hex(openssl_public_key_decoded.y)}")

In [ ]:
message = b"Hello, ECDSA!"

# Sign the message with the Private Key
signature = openssl_private_key.sign(message, ec.ECDSA(hashes.SHA256()))

print(f"Message: {message.decode()}")
print(f"Signature (Hex): {signature.hex()}\n")

# Verify the signature with the Public Key
try:
    openssl_public_key.verify(signature, message, ec.ECDSA(hashes.SHA256()))
    print("✅ Verification Successful! The signature matches the message and public key.")
except InvalidSignature:
    print("❌ Verification Failed! The signature is invalid, or the message was tampered with.")

In [ ]:
def find_inverse(number, modulus):
    return pow(number, -1, modulus)

class Point:
    def __init__(self, x, y, curve_config):
        a = curve_config['a']
        b = curve_config['b']
        p = curve_config['p']

        if (y ** 2) % p != (x ** 3 + a * x + b) % p:
            raise Exception("The point is not on the curve")

        self.x = x
        self.y = y
        self.curve_config = curve_config

    def is_equal_to(self, point):
        return self.x == point.x and self.y == point.y

    def add(self, point):
        p = self.curve_config['p']

        if self.is_equal_to(point):
            slope = (((3 * point.x ** 2) + self.curve_config['a'])) * find_inverse(2 * point.y, p) % p
        else:
            slope = (point.y - self.y) * find_inverse(point.x - self.x, p) % p
            
        x = (slope ** 2 - point.x - self.x) % p
        y = (slope * (self.x - x) - self.y) % p
        return Point(x, y, self.curve_config)

    def multiply(self, times):
        current_point = self
        current_coefficient = 1

        pervious_points = []
        while current_coefficient < times:
            # Store current point as a previous point
            pervious_points.append((current_coefficient, current_point))
            # If we can multiply our current point by 2, do it
            if 2 * current_coefficient <= times:
                current_point = current_point.add(current_point)
                current_coefficient = 2 * current_coefficient
            # If we can't multiply our current point by 2, let's find the biggest previous point to add to our point
            else:
                next_point = self
                next_coefficient = 1
                for (previous_coefficient, previous_point) in pervious_points:
                    if previous_coefficient + current_coefficient <= times:
                        if previous_point.x != current_point.x:
                            next_coefficient = previous_coefficient
                            next_point = previous_point
                current_point = current_point.add(next_point)
                current_coefficient = current_coefficient + next_coefficient

        return current_point

secp256r1_curve_config = {
    'a': int("FFFFFFFF_00000001_00000000_00000000_00000000_FFFFFFFF_FFFFFFFF_FFFFFFFC", 16),
    'b': int("5AC635D8_AA3A93E7_B3EBBD55_769886BC_651D06B0_CC53B0F6_3BCE3C3E_27D2604B", 16),
    'p': int("FFFFFFFF_00000001_00000000_00000000_00000000_FFFFFFFF_FFFFFFFF_FFFFFFFF", 16)
}
BASE_POINT_Gx = int("6B17D1F2_E12C4247_F8BCE6E5_63A440F2_77037D81_2DEB33A0_F4A13945_D898C296", 16)
BASE_POINT_Gy = int("4FE342E2_FE1A7F9B_8EE7EB4A_7C0F9E16_2BCE3357_6B315ECE_CBB64068_37BF51F5", 16)
n = int("FFFFFFFF_00000000_FFFFFFFF_FFFFFFFF_BCE6FAAD_A7179E84_F3B9CAC2_FC632551", 16)
g_point = Point(BASE_POINT_Gx, BASE_POINT_Gy, secp256r1_curve_config)

def sign_message(message, private_key):
    k = random.randint(1, n)
    r_point = g_point.multiply(k)
    r = r_point.x % n
    if r == 0:
        return sign_message(message, private_key)
    k_inverse = find_inverse(k, n)
    s = (message + (r * private_key)) * k_inverse % n
    return r, s

def verify_signature(signature, message, public_key):
    (r, s) = signature
    s_inverse = find_inverse(s, n)
    u = message * s_inverse % n
    v = r * s_inverse % n
    c_point = g_point.multiply(u).add(public_key.multiply(v))
    return c_point.x == r

In [ ]:
public_key = g_point.multiply(openssl_private_key_decoded)

print("OPENSSL_REF_PRIVATE_KEY:", f"{openssl_private_key_decoded:X}")
if openssl_public_key_decoded.x == public_key.x and openssl_public_key_decoded.y == public_key.y:
	print("OPENSSL_REF_PUBLIC_KEY_X:", f"{openssl_public_key_decoded.x:X}")
	print("OPENSSL_REF_PUBLIC_KEY_Y:", f"{openssl_public_key_decoded.y:X}")
	print("COMPUTED_PUBLIC_KEY_X:", f"{public_key.x:X}")
	print("COMPUTED_PUBLIC_KEY_Y:", f"{public_key.y:X}")
	print(True)

In [ ]:
data = "Hello, ECDSA!"
sha256_hash = hashlib.sha256(data.encode('utf-8')).hexdigest()
signature = sign_message(int(sha256_hash, 16), openssl_private_key_decoded)
print('Signature: ', hex(signature[0]), hex(signature[1]))
print('Is valid: ', verify_signature(signature, int(sha256_hash, 16), public_key))

In [ ]:
def point_addition(p_QX, p_QY, p_RX, p_RY):
    slope = (p_RY - p_QY) * find_inverse(p_RX - p_QX, secp256r1_curve_config['p']) % secp256r1_curve_config['p']
    x = (slope ** 2 - p_QX - p_RX) % secp256r1_curve_config['p']
    y = (slope * (p_QX - x) - p_QY) % secp256r1_curve_config['p']
    return x, y

def point_doubling(p_RX, p_RY):
    slope = (((3 * p_RX ** 2) + secp256r1_curve_config['a'])) * find_inverse(2 * p_RY, secp256r1_curve_config['p']) % secp256r1_curve_config['p']
    x = (slope ** 2 - p_RX - p_RX) % secp256r1_curve_config['p']
    y = (slope * (p_RX - x) - p_RY) % secp256r1_curve_config['p']
    return x, y

In [ ]:
CURVE_OPERATION_FILE_PATH = 'SCALAR_POINT_MULTIPLICATION_OP_REF.csv'

QX = 0
QY = 0

k  = openssl_private_key_decoded

RX = BASE_POINT_Gx
RY = BASE_POINT_Gy

header = ['Iteration', 'k[0]', 'k', 'QX', 'QY', 'RX', 'RY', 'POINT_ADDITION', 'POINT_DOUBLING']

iteration = 0

with open(CURVE_OPERATION_FILE_PATH, 'w', newline = '') as file:
    writer = csv.writer(file)
    writer.writerow(header)

while (k != 0):
    # print("K[0]: ", k & 1)
    data = []
    data.append(iteration)
    data.append(k & 1)
    data.append(k)
    PA_Hit = 0
    PD_Hit = 0
    if (k & 1) == 1:
        if QX == 0 and QY == 0:
            QX = RX
            QY = RY
        else:
            QX, QY = point_addition(QX, QY, RX, RY)
            PA_Hit = 1
            print("Point_Addition: QX, QY", f"{QX:X}", f"{QY:X}")
    RX, RY = point_doubling(RX, RY)
    PD_Hit = 1
    print("Point_Doubling: RX, RY", f"{RX:X}", f"{RY:X}")
    k = k >> 1
    data.append(QX)
    data.append(QY)
    data.append(RX)
    data.append(RY)
    if PA_Hit == 1: 
        data.append("True")
    else:
        data.append("False")
    if PD_Hit == 1: 
        data.append("True")
    else:
        data.append("False")
    for i in range(2, 7):
        data[i] = f"{data[i]:X}"
    with open(CURVE_OPERATION_FILE_PATH, 'a', newline = '') as file:
        writer = csv.writer(file)
        writer.writerow(data)
    iteration = iteration + 1

if openssl_public_key_decoded.x == QX and openssl_public_key_decoded.y == QY:
    print("OPENSSL_REF_PRIVATE_KEY:", f"{openssl_private_key_decoded:X}")
    print("COMPUTED_PUBLIC_KEY_X (QX):", f"{QX:X}")
    print("COMPUTED_PUBLIC_KEY_Y (QY):", f"{QY:X}")
    print("OPENSSL_REF_PUBLIC_KEY_X:", f"{openssl_public_key_decoded.x:X}")
    print("OPENSSL_REF_PUBLIC_KEY_Y:", f"{openssl_public_key_decoded.y:X}")
    print(True)